In [1]:
import os
import json
import re
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import numpy as np
import pprint

from typing import List, Dict, Any, Tuple
from collections import defaultdict, Counter
from utils import count_relaxed_matches, count_exact_matches, exact_match_results, relax_match_results, sample_to_prediction_list


/home/hieu/miniconda3/envs/ros/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to /home/hieu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# model_folder = "Output_llama3.1_8b_no_sim"
# model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_26b_no_sim"
model_folder = "Output_gemma4_31b_no_sim"

In [3]:
if os.path.exists(f"results/Output_verification_em.csv") == True:
    os.remove(f"results/Output_verification_em.csv")
if os.path.exists(f"results/Output_verification_rm.csv") == True:
    os.remove(f"results/Output_verification_rm.csv")


tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_38_20210610044413_Hayden:
[{'abdominal pain': 'PROBLEM'}, {'weight loss': 'PROBLEM'}, {'jaundice': 'PROBLEM'}, {'epigastric pain': 'PROBLEM'}, {'thin-slice CT scan': 'TEST'}, {'pancreatic mass': 'PROBLEM'}, {'involved lymph nodes': 'PROBLEM'}, {'ring enhancing lesions': 'PROBLEM'}, {'liver metastases': 'PROBLEM'}, {'abdominal pain': 'PROBLEM'}, {'weight loss': 'PROBLEM'}, {'jaundice': 'PROBLEM'}, {'epigastric pain': 'PROBLEM'}, {'thin - slice CT scan': 'TEST'}, {'pancreatic mass': 'PROBLEM'}, {'involved lymph nodes': 'PROBLEM'}, {'ring enhancing lesions': 'PROBLEM'}, {'liver metastases': 'PROBLEM'}, {'pseudocyst': 'PROBLEM'}, {'ERCP': 'TEST'}, {'placement of a stent': 'TREATMENT'}, {'strictured pancreatic duct': 'PROBLEM'}, {'strictured bile duct': 'PROBLEM'}, {'10-frenchx9 cm stent': 'TREATMENT'}, {'bilirubin': 'TEST'}, {'liver tests': 'TEST'}, {'nausea': 'PROBLEM'}, {'Zofran': 'DRUG'}, {'Phenergan': 'DRUG'}, {'nausea': 'PROBLEM'}, {'Phenergan': 'DRUG'}, {'white coun

In [4]:
df = pd.read_csv('results/Output_verification_rm.csv',header=None)
df.columns = ['file','RM','MM','UD','OD','Prec','Sens','F1']
print(df['F1'].mean())

df.sort_values(by='OD',ascending=False).head(10)

0.8316438848920863


,file,RM,MM,UD,OD,Prec,Sens,F1
104,sample_635,99,8,24,29,0.7279,0.8049,0.7645
105,sample_641,191,24,12,29,0.7828,0.9409,0.8546
96,sample_82_20210604143826_Booma,216,9,27,22,0.8745,0.8889,0.8816
87,sample_2748,109,2,20,21,0.8258,0.8450,0.8352
101,sample_1258,143,5,40,19,0.8563,0.7814,0.8171
63,sample_74_20210610044426_Booma,54,6,40,18,0.6923,0.5745,0.6279
24,sample_2_20210604143805_Hayden,18,0,5,17,0.5143,0.7826,0.6207
119,sample_128_20210112211711,74,11,6,16,0.7327,0.9250,0.8177
8,sample_1262,141,4,20,14,0.8868,0.8758,0.8812
91,sample_91_20210112211706,166,17,11,14,0.8426,0.9379,0.8877


# Further analysis